# Olist Feature Engineering

**Goal:** use checkout-time information to predict an item's recorded `freight_value` (R$). This is a regression exercise, not a complete production shipping-quote system.

**One row:** one order item, identified by `order_id` and `order_item_id`.

**Prediction moment:** checkout. Use only information available then, not delivery outcomes or reviews.

Run root `scripts/bootstrap.py` once, then run these cells from within the cloned repository. The Olist CSVs are in the shared `datasets/olist/Ecommerce Dataset/` folder.

## 1. Import libraries and find the data

We only load tables needed for the item features in this lesson.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'datasets' / 'olist' / 'Ecommerce Dataset.zip').is_file()), None)
assert ROOT is not None, 'Open the notebook from inside the cloned course repository.'
DATA = ROOT / 'datasets' / 'olist' / 'Ecommerce Dataset'
assert DATA.is_dir(), 'Run scripts/bootstrap.py first to extract the Olist CSVs.'
print('Dataset folder:', DATA.resolve())

## 2. Load the tables we need

Payments and reviews are intentionally left out: they are not checkout-time predictors here, and each can contain multiple rows per order. ZIP prefixes are read as text so joins keep their formatting.

In [ ]:
orders = pd.read_csv(
    DATA / 'olist_orders_dataset.csv',
    usecols=['order_id', 'customer_id', 'order_purchase_timestamp'],
    parse_dates=['order_purchase_timestamp'],
)
items = pd.read_csv(DATA / 'olist_order_items_dataset.csv')
customers = pd.read_csv(
    DATA / 'olist_customers_dataset.csv',
    dtype={'customer_zip_code_prefix': 'string'},
)
sellers = pd.read_csv(
    DATA / 'olist_sellers_dataset.csv',
    dtype={'seller_zip_code_prefix': 'string'},
)
products = pd.read_csv(
    DATA / 'olist_products_dataset.csv',
    usecols=[
        'product_id', 'product_category_name', 'product_weight_g',
        'product_length_cm', 'product_height_cm', 'product_width_cm',
    ],
)
geo = pd.read_csv(
    DATA / 'olist_geolocation_dataset.csv',
    usecols=['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng'],
    dtype={'geolocation_zip_code_prefix': 'string'},
)
translation = pd.read_csv(DATA / 'product_category_name_translation.csv')

tables = {'orders': orders, 'items': items, 'customers': customers,
          'sellers': sellers, 'products': products, 'geolocation': geo}
display(pd.DataFrame({name: table.shape for name, table in tables.items()},
                     index=['rows', 'columns']).T)

## 3. Understand the row and check the target

Items are the base table: one row per (`order_id, order_item_id`). Orders, customers, sellers and products should each match at most one row to an item. Geolocation, payments and reviews can have repeated keys, so they must not be joined raw.

In [ ]:
print('Item rows:', len(items))
print('Unique item keys:', len(items.drop_duplicates(['order_id', 'order_item_id'])))
print('Missing freight targets:', items['freight_value'].isna().sum())
print('Negative freight values:', items['freight_value'].lt(0).sum())
assert not items.duplicated(['order_id', 'order_item_id']).any()
assert items['freight_value'].notna().all()

## 4. Split by purchase date

Reserve later orders as validation and test data. This better represents predicting the future than a random split. Do target-based exploration on training rows only.

In [ ]:
dated_items = items.merge(
    orders[['order_id', 'order_purchase_timestamp']],
    on='order_id', validate='many_to_one',
)

train_raw = dated_items[dated_items.order_purchase_timestamp < '2018-04-01'].copy()
validation_raw = dated_items[dated_items.order_purchase_timestamp.between(
    '2018-04-01', '2018-06-01', inclusive='left'
)].copy()
test_raw = dated_items[dated_items.order_purchase_timestamp.between(
    '2018-06-01', '2018-09-01', inclusive='left'
)].copy()

print('Train:', len(train_raw), '| validation:', len(validation_raw),
      '| test:', len(test_raw))
assert set(train_raw.order_id).isdisjoint(validation_raw.order_id)
assert set(train_raw.order_id).isdisjoint(test_raw.order_id)
assert set(validation_raw.order_id).isdisjoint(test_raw.order_id)

## 5. Look at the target

Freight is a numeric target, so inspect its distribution rather than converting it into an arbitrary high/low label. Keep expensive but valid examples; limit the plot only if needed for readability.

In [ ]:
display(train_raw['freight_value'].describe(percentiles=[.5, .9, .99]).round(2))

plt.figure(figsize=(8, 4))
plt.hist(train_raw['freight_value'], bins=40, edgecolor='white')
plt.xlabel('Freight value (R$)')
plt.ylabel('Number of items')
plt.title('Training freight values')
plt.tight_layout()
plt.show()

## 6. A quick time check

This simple summary shows why time matters and gives a reason for the chronological split. It is descriptive, not a causal explanation.

In [ ]:
monthly = train_raw.set_index('order_purchase_timestamp').resample('MS')['freight_value'].median()
monthly.plot(marker='o', figsize=(8, 4))
plt.ylabel('Median freight (R$)')
plt.title('Monthly median freight during training')
plt.tight_layout()
plt.show()

## 7. Join the item, order and lookup tables

Use left joins to retain each item. `validate='many_to_one'` checks that lookup keys do not duplicate item rows.

In [ ]:
df = items.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
    on='order_id', validate='many_to_one',
).merge(
    customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']],
    on='customer_id', validate='many_to_one',
).merge(
    sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']],
    on='seller_id', validate='many_to_one',
).merge(
    products, on='product_id', how='left', validate='many_to_one',
).merge(
    translation, on='product_category_name', how='left', validate='many_to_one',
)

assert len(df) == len(items)
print('Rows after lookup joins:', len(df))

## 8. Make one coordinate per ZIP prefix

The geolocation table has many coordinates for the same ZIP prefix. Use the median coordinate for each prefix before joining; this is an approximate location, not an exact address.

In [ ]:
geo_in_brazil = (
    geo.geolocation_lat.between(-34, 6)
    & geo.geolocation_lng.between(-74, -34)
)
zip_coordinates = geo.loc[geo_in_brazil].groupby(
    'geolocation_zip_code_prefix'
)[['geolocation_lat', 'geolocation_lng']].median()

df = df.join(zip_coordinates.add_prefix('customer_'),
             on='customer_zip_code_prefix')
df = df.join(zip_coordinates.add_prefix('seller_'),
             on='seller_zip_code_prefix')

assert len(df) == len(items)
print('Rows with missing customer coordinates:',
      df.customer_geolocation_lat.isna().sum())
print('Rows with missing seller coordinates:',
      df.seller_geolocation_lat.isna().sum())

## 9. Create a few meaningful features

Build features from physical size, approximate route, location, basket composition and purchase date. These are intended to be known at checkout.

In [ ]:
# Approximate straight-line distance between ZIP-prefix coordinates
lat1 = np.radians(df.seller_geolocation_lat)
lon1 = np.radians(df.seller_geolocation_lng)
lat2 = np.radians(df.customer_geolocation_lat)
lon2 = np.radians(df.customer_geolocation_lng)
delta_lat = lat2 - lat1
delta_lon = lon2 - lon1
a = (np.sin(delta_lat / 2) ** 2
     + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2)
df['distance_km'] = 2 * 6371 * np.arcsin(np.sqrt(a.clip(0, 1)))

# Product, location, basket and calendar features
df['volume_cm3'] = (df.product_length_cm * df.product_height_cm
                    * df.product_width_cm)
df['same_state'] = (df.seller_state == df.customer_state).astype(int)
df['items_in_order'] = df.groupby('order_id').order_item_id.transform('size')
df['items_from_seller'] = df.groupby(
    ['order_id', 'seller_id']
).order_item_id.transform('size')
df['purchase_month'] = df.order_purchase_timestamp.dt.month
df['product_category'] = (
    df.product_category_name_english
    .fillna(df.product_category_name)
    .fillna('unknown')
)

print('Created distance, volume, state, basket, month, and category features.')

## 9. Create a few meaningful features

Build features from product size, approximate route, location, basket composition and purchase date. Basket counts assume the complete checkout basket is known. Volume uses catalog dimensions; distance uses straight-line ZIP-prefix coordinates, not road distance.

In [ ]:
train_eda = df[df.order_purchase_timestamp < '2018-04-01']

print('Median freight by same-state flag:')
display(train_eda.groupby('same_state').freight_value.median().round(2))

weight_band = pd.cut(
    train_eda.product_weight_g,
    bins=[0, 500, 2000, 10000, np.inf],
    labels=['under 500 g', '500 g to 2 kg', '2 to 10 kg', 'over 10 kg'],
)
print('Median freight by product weight:')
display(train_eda.groupby(weight_band, observed=True).freight_value
        .agg(['count', 'median']).round(2))

## 11. Visualize distance and freight

A sample keeps the scatter plot readable. Distance is approximate; a relationship does not prove cause and effect.

In [ ]:
plot_data = train_eda[['distance_km', 'freight_value']].dropna()
plot_data = plot_data.sample(n=min(3000, len(plot_data)), random_state=42)
plt.figure(figsize=(7, 4))
plt.scatter(plot_data.distance_km, plot_data.freight_value, alpha=0.25)
plt.xlabel('Approximate distance (km)')
plt.ylabel('Freight (R$)')
plt.title('Training items: distance vs freight')
plt.tight_layout()
plt.show()

## 12. Check missing values; do not treat missing as zero

Keep rows with incomplete product dimensions or coordinates. Impute numeric values using training data only. An `unknown` category is an explicit category, not a missing number.

In [ ]:
feature_check = ['price', 'product_weight_g', 'product_length_cm',
                 'product_height_cm', 'product_width_cm', 'volume_cm3',
                 'distance_km', 'product_category', 'freight_value']
missing = train_eda[feature_check].isna().sum()
display(missing[missing > 0].to_frame('missing training rows'))
print('Unknown training categories:',
      train_eda.product_category.eq('unknown').sum())

## 13. Choose features and keep leakage out

IDs and timestamps help with joining and splitting, but are not predictors. Exclude the target, delivery outcomes, reviews and other values unavailable at checkout.

In [ ]:
categorical = ['seller_state', 'customer_state', 'product_category']
numeric = [
    'price', 'product_weight_g', 'product_length_cm', 'product_height_cm',
    'product_width_cm', 'volume_cm3', 'distance_km', 'same_state',
    'items_in_order', 'items_from_seller', 'purchase_month',
]
features = categorical + numeric
model_data = df[['order_id', 'order_item_id', 'order_purchase_timestamp',
                 'freight_value'] + features].copy()

train = model_data[model_data.order_purchase_timestamp < '2018-04-01']
validation = model_data[model_data.order_purchase_timestamp.between(
    '2018-04-01', '2018-06-01', inclusive='left'
)]
test = model_data[model_data.order_purchase_timestamp.between(
    '2018-06-01', '2018-09-01', inclusive='left'
)]

X_train, y_train = train[features], train.freight_value
X_val, y_val = validation[features], validation.freight_value
X_test, y_test = test[features], test.freight_value

assert len(model_data) == len(items)
assert 'freight_value' not in features
assert set(train.order_id).isdisjoint(validation.order_id)
assert set(train.order_id).isdisjoint(test.order_id)
assert set(validation.order_id).isdisjoint(test.order_id)
print(f'Train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
print('Predictor columns:', len(features))

## 14. Prepare preprocessing for the next lesson

The preprocessor is defined but not fitted. In the model-training lesson, fit it inside a pipeline using training rows only; then apply the same learned steps to validation and test data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler()),
    ]), numeric),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical),
])
print('Preprocessor ready; not fitted.')

## Exit ticket

1. What does one row represent?
2. Why do we aggregate geolocation before joining?
3. Explain `volume_cm3` and `distance_km`. What are their limitations?
4. Which columns are unavailable at checkout and must be excluded?
5. Why must missing-value handling and encoding be learned from training data only?

**Next:** establish a median baseline, train a regression model, and compare validation MAE in R$. Use the test set only for final evaluation.

## 15. Export the processed dataset

Save one row per order item with the target, item keys, purchase date, feature columns and time split. The split label makes it clear which rows are for training, validation, testing, or outside the defined date windows. Do not use the IDs, date, split label, or target as predictors.

In [ ]:
final_dataset = model_data.copy()
final_dataset['data_split'] = 'outside_defined_windows'
final_dataset.loc[train.index, 'data_split'] = 'train'
final_dataset.loc[validation.index, 'data_split'] = 'validation'
final_dataset.loc[test.index, 'data_split'] = 'test'

output_path = ROOT / 'exercises' / '01-eda-feature-engineering' / 'outputs' / 'olist_freight_features.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
final_dataset.to_csv(output_path, index=False)

assert len(final_dataset) == len(items)
assert not final_dataset.duplicated(['order_id', 'order_item_id']).any()
print(f'Exported {len(final_dataset):,} rows to: {output_path.resolve()}')
display(final_dataset.head())